# 🌋 SEISMEX - Demo Completa

Este notebook demuestra las capacidades principales del sistema SEISMEX para análisis de riesgo sísmico en México.

## Contenido

1. [Configuración inicial](#1-configuración-inicial)
2. [Análisis Gutenberg-Richter](#2-análisis-gutenberg-richter)
3. [Generación de Isosistas](#3-generación-de-isosistas)
4. [Modelos de Fuentes Sísmicas](#4-modelos-de-fuentes-sísmicas)
5. [Análisis PSHA](#5-análisis-psha)
6. [Optimización NSGA-II](#6-optimización-nsga-ii)
7. [Pipeline Integrado](#7-pipeline-integrado)

## 1. Configuración inicial

In [ ]:
# Importaciones básicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Importar módulos de SEISMEX
from seismex.analysis import (
    # Isosistas
    GeneradorIsosistas,
    crear_generador_mexico,
    GMPEGarcia2005,
    IPECENAPRED2006,
    
    # Modelos de fuentes
    ModeloFuentes,
    FuenteArea,
    crear_modelo_mexico_simplificado,
    DistribucionGutenbergRichter,
    
    # PSHA
    AnalizadorPSHA,
    crear_analizador_mexico,
    CurvaPeligro,
)

print("✅ Módulos de SEISMEX cargados correctamente")

## 2. Análisis Gutenberg-Richter

Demostración del análisis de la relación frecuencia-magnitud.

In [ ]:
# Crear datos sintéticos de ejemplo (en producción usarías CatalogoSismico)
np.random.seed(42)

# Simular catálogo con distribución G-R
b_value = 1.0
a_value = 5.0
mmin, mmax = 3.0, 7.5

# Generar magnitudes siguiendo G-R
n_eventos = 5000
u = np.random.rand(n_eventos)
magnitudes = mmin - np.log10(1 - u * (1 - 10**(-b_value * (mmax - mmin)))) / b_value

# Visualizar distribución
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(magnitudes, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Magnitud')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Magnitudes')

# Relación G-R
bins = np.arange(mmin, mmax + 0.1, 0.1)
counts, _ = np.histogram(magnitudes, bins=bins)
cumulative = np.cumsum(counts[::-1])[::-1]

axes[1].semilogy(bins[:-1], cumulative, 'ko', markersize=5, label='Datos')

# Ajuste teórico
m_fit = np.linspace(mmin, mmax, 100)
n_fit = 10**(a_value - b_value * m_fit)
axes[1].semilogy(m_fit, n_fit, 'r-', linewidth=2, label=f'G-R: b={b_value:.2f}')

axes[1].set_xlabel('Magnitud')
axes[1].set_ylabel('N acumulado (≥M)')
axes[1].set_title('Relación Gutenberg-Richter')
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Eventos simulados: {n_eventos}")
print(f"Magnitud mínima: {magnitudes.min():.2f}")
print(f"Magnitud máxima: {magnitudes.max():.2f}")

## 3. Generación de Isosistas

Cálculo de mapas de intensidad sísmica usando GMPEs e IPEs.

In [ ]:
# Crear generador de isosistas para México
generador = crear_generador_mexico()

print(f"IPE: {generador.ipe.nombre}")
print(f"GMPE: {generador.gmpe.nombre}")

In [ ]:
# Definir evento sísmico (Sismo de Colima 2003, M7.6)
evento = {
    'latitud': 18.71,
    'longitud': -104.13,
    'profundidad_km': 24,
    'magnitud': 7.6
}

# Calcular isosistas
isosistas = generador.calcular(
    latitud=evento['latitud'],
    longitud=evento['longitud'],
    profundidad_km=evento['profundidad_km'],
    magnitud=evento['magnitud'],
    resolucion_km=5.0,
    radio_max_km=400
)

print(f"\nResultados:")
print(f"  Intensidad máxima: {isosistas.intensidad_maxima:.1f} MMI")
print(f"  Intensidad en epicentro: {isosistas.intensidad_epicentro:.1f} MMI")
print(f"  Dimensiones del grid: {isosistas.intensidad_grid.shape}")

In [ ]:
# Visualizar isosistas
fig, ax = plt.subplots(figsize=(12, 10))

isosistas.graficar(
    ax=ax,
    mostrar_epicentro=True,
    mostrar_contornos=True,
    colorbar=True,
    titulo=f"Isosistas - Sismo de Colima M{evento['magnitud']}"
)

plt.tight_layout()
plt.show()

In [ ]:
# Comparar diferentes IPEs
from seismex.analysis import IPEAllen2012, IPECENAPRED2006

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# IPE Allen 2012
gen_allen = GeneradorIsosistas(ipe='allen_2012')
iso_allen = gen_allen.calcular(**evento, resolucion_km=8, radio_max_km=300)
iso_allen.graficar(ax=axes[0], titulo='Allen et al. (2012)')

# IPE CENAPRED
gen_cenapred = GeneradorIsosistas(ipe='cenapred_2006')
iso_cenapred = gen_cenapred.calcular(**evento, resolucion_km=8, radio_max_km=300)
iso_cenapred.graficar(ax=axes[1], titulo='CENAPRED (2006)')

plt.tight_layout()
plt.show()

print(f"\nIntensidad máxima:")
print(f"  Allen 2012: {iso_allen.intensidad_maxima:.1f} MMI")
print(f"  CENAPRED 2006: {iso_cenapred.intensidad_maxima:.1f} MMI")

## 4. Modelos de Fuentes Sísmicas

Creación y uso de modelos de fuentes para PSHA.

In [ ]:
# Crear modelo de fuentes simplificado para México
modelo = crear_modelo_mexico_simplificado()

print(modelo.resumen())

In [ ]:
# Crear un modelo personalizado
from seismex.analysis import TipoFalla

mi_modelo = ModeloFuentes(
    nombre="Modelo Colima",
    descripcion="Modelo simplificado de fuentes para la región de Colima",
    version="1.0"
)

# Agregar zona de subducción
mi_modelo.agregar_zona_area(
    nombre="Subducción Jalisco-Colima",
    poligono=[
        (17.5, -106.0), (19.0, -105.0), (19.5, -103.5),
        (18.5, -102.5), (17.0, -104.0), (17.0, -105.5)
    ],
    a_value=4.8,
    b_value=0.95,
    mmin=5.0,
    mmax=8.2,
    profundidad_media=30
)

# Agregar zona de sismicidad cortical
mi_modelo.agregar_zona_area(
    nombre="Graben de Colima",
    poligono=[
        (19.0, -104.0), (19.8, -103.8), (20.0, -103.0),
        (19.5, -102.8), (19.0, -103.2)
    ],
    a_value=3.5,
    b_value=1.1,
    mmin=4.0,
    mmax=6.5,
    profundidad_media=10
)

print(mi_modelo.resumen())

In [ ]:
# Generar catálogo sintético
catalogo_sintetico = mi_modelo.muestrear_catalogo(n_eventos=500, seed=42)

# Convertir a DataFrame para visualización
df = pd.DataFrame(catalogo_sintetico)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Mapa de epicentros
scatter = axes[0].scatter(
    df['lon'], df['lat'],
    c=df['magnitud'], s=df['magnitud']**2,
    cmap='YlOrRd', alpha=0.6, edgecolors='black', linewidth=0.5
)
plt.colorbar(scatter, ax=axes[0], label='Magnitud')
axes[0].set_xlabel('Longitud')
axes[0].set_ylabel('Latitud')
axes[0].set_title('Epicentros (Catálogo Sintético)')
axes[0].set_aspect('equal')

# Histograma de magnitudes
axes[1].hist(df['magnitud'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].set_xlabel('Magnitud')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Magnitudes')

plt.tight_layout()
plt.show()

print(f"\nEstadísticas del catálogo sintético:")
print(df.describe())

## 5. Análisis PSHA

Análisis Probabilístico de Peligro Sísmico con metodología Cornell-McGuire.

In [ ]:
# Crear analizador PSHA preconfigurado para México
psha = crear_analizador_mexico(vs30=400)

print(psha.resumen())

In [ ]:
# Calcular curva de peligro para Ciudad de México
sitio_cdmx = (19.4326, -99.1332)

curva_cdmx = psha.calcular_curva_peligro(
    sitio=sitio_cdmx,
    vs30=350  # Suelo blando típico de CDMX
)

# Obtener PGA para diferentes períodos de retorno
trs = [72, 225, 475, 975, 2475]
print("\nPGA para diferentes períodos de retorno (Ciudad de México):")
print("-" * 50)
for tr in trs:
    pga = curva_cdmx.intensidad_para_periodo_retorno(tr)
    prob = curva_cdmx.probabilidad_excedencia(pga, tiempo_exposicion=50)
    print(f"  TR = {tr:4d} años: PGA = {pga:.3f} g ({prob*100:.1f}% en 50 años)")

In [ ]:
# Comparar curvas de peligro para diferentes ciudades
ciudades = {
    'Ciudad de México': (19.4326, -99.1332, 350),
    'Guadalajara': (20.6597, -103.3496, 500),
    'Acapulco': (16.8531, -99.8237, 400),
    'Oaxaca': (17.0732, -96.7266, 450),
}

fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.Set1(np.linspace(0, 1, len(ciudades)))

for (ciudad, (lat, lon, vs30)), color in zip(ciudades.items(), colors):
    curva = psha.calcular_curva_peligro(sitio=(lat, lon), vs30=vs30)
    ax.loglog(curva.intensidades, curva.tasas_excedencia, 
              linewidth=2, label=ciudad, color=color)

# Líneas de período de retorno
for tr in [475, 2475]:
    ax.axhline(1/tr, color='gray', linestyle='--', alpha=0.5)
    ax.text(0.5, 1/tr * 1.2, f'TR={tr} años', fontsize=9, color='gray')

ax.set_xlabel('PGA (g)', fontsize=12)
ax.set_ylabel('Tasa de excedencia anual', fontsize=12)
ax.set_title('Curvas de Peligro Sísmico - Ciudades de México', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(0.001, 2)
ax.set_ylim(1e-5, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Desagregación del peligro
desag = psha.desagregar(
    sitio=sitio_cdmx,
    nivel_intensidad=0.15,  # PGA = 0.15g
    vs30=350
)

print(desag.resumen())

In [ ]:
# Visualizar desagregación M-R
fig, ax = plt.subplots(figsize=(10, 8))

desag.graficar_MR(ax=ax)

plt.tight_layout()
plt.show()

## 6. Optimización NSGA-II

Optimización multiobjetivo para ubicación de infraestructura.

In [ ]:
from seismex.optimization import (
    OptimizadorNSGAII,
    ConfiguracionNSGAII,
    objetivo_costo_construccion,
    objetivo_accesibilidad,
    restriccion_distancia_minima,
)

# Configurar optimizador
config = ConfiguracionNSGAII(
    n_generaciones=50,
    tamano_poblacion=100,
    n_sitios=3,  # Buscar 3 ubicaciones óptimas
    prob_cruce=0.9,
    prob_mutacion=0.1
)

print(f"Configuración NSGA-II:")
print(f"  Generaciones: {config.n_generaciones}")
print(f"  Población: {config.tamano_poblacion}")
print(f"  Sitios a optimizar: {config.n_sitios}")

In [ ]:
# Crear optimizador
optimizador = OptimizadorNSGAII(config)

# Definir puntos de interés (ciudades principales)
pois = [
    (19.5, -103.5),  # Colima
    (20.7, -103.3),  # Guadalajara
    (19.0, -104.0),  # Manzanillo
]

# Agregar objetivos
optimizador.agregar_objetivo(objetivo_costo_construccion())
optimizador.agregar_objetivo(objetivo_accesibilidad(pois))

# Agregar restricciones
optimizador.agregar_restriccion(restriccion_distancia_minima(distancia_km=30))

print(f"\nObjetivos: {len(optimizador.objetivos)}")
print(f"Restricciones: {len(optimizador.restricciones)}")

In [ ]:
# Ejecutar optimización
print("Ejecutando optimización NSGA-II...")

resultado = optimizador.optimizar(
    bounds=[(18.5, 21.0), (-105.0, -102.0)]  # Región de búsqueda
)

print(f"\n{resultado.resumen()}")

In [ ]:
# Visualizar frente de Pareto
fig, ax = plt.subplots(figsize=(10, 8))

resultado.graficar_pareto(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Obtener solución de compromiso
solucion = resultado.obtener_solucion_compromiso()

print("Solución de compromiso:")
print(f"  Coordenadas: {solucion.decodificar_coordenadas()}")
print(f"  Valores objetivo: {solucion.valores_objetivo}")

## 7. Pipeline Integrado

Ejemplo de flujo de trabajo completo: desde fuentes hasta mapas de peligro.

In [ ]:
# Pipeline completo para análisis de peligro sísmico

# 1. Definir región de estudio
region = {
    'nombre': 'Occidente de México',
    'lat': (17, 22),
    'lon': (-106, -101)
}

# 2. Crear modelo de fuentes
fuentes = crear_modelo_mexico_simplificado()
print(f"✓ Modelo de fuentes: {len(fuentes)} fuentes")

# 3. Configurar PSHA
psha = AnalizadorPSHA(
    fuentes=fuentes,
    vs30=400,
    distancia_maxima=400
)
psha.agregar_gmpe(GMPEGarcia2005(), peso=0.6)

from seismex.analysis import GMPEZhao2006
psha.agregar_gmpe(GMPEZhao2006(), peso=0.4)
print(f"✓ PSHA configurado con {len(psha.gmpes)} GMPEs")

# 4. Calcular curvas para sitios de interés
sitios = {
    'Colima': (19.24, -103.72),
    'Puerto Vallarta': (20.62, -105.23),
    'Manzanillo': (19.05, -104.32),
}

resultados = {}
for nombre, (lat, lon) in sitios.items():
    curva = psha.calcular_curva_peligro(sitio=(lat, lon))
    pga_475 = curva.intensidad_para_periodo_retorno(475)
    resultados[nombre] = pga_475
    print(f"  {nombre}: PGA(475) = {pga_475:.3f} g")

print("\n✓ Pipeline completado")

In [ ]:
# Visualización final
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Curvas de peligro
ax1 = axes[0, 0]
for nombre, (lat, lon) in sitios.items():
    curva = psha.calcular_curva_peligro(sitio=(lat, lon))
    ax1.loglog(curva.intensidades, curva.tasas_excedencia, linewidth=2, label=nombre)
ax1.axhline(1/475, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('PGA (g)')
ax1.set_ylabel('Tasa de excedencia anual')
ax1.set_title('Curvas de Peligro')
ax1.legend()
ax1.grid(True, which='both', alpha=0.3)

# 2. Barras de PGA
ax2 = axes[0, 1]
nombres = list(resultados.keys())
valores = list(resultados.values())
bars = ax2.bar(nombres, valores, color='steelblue', edgecolor='black')
ax2.set_ylabel('PGA (g)')
ax2.set_title('PGA para TR=475 años')
ax2.bar_label(bars, fmt='%.3f')

# 3. Isosistas de escenario
ax3 = axes[1, 0]
gen = crear_generador_mexico()
iso = gen.calcular(latitud=18.7, longitud=-104.1, profundidad_km=25, magnitud=7.5,
                   resolucion_km=10, radio_max_km=250)
iso.graficar(ax=ax3, titulo='Escenario M7.5')

# 4. Mapa de ubicaciones
ax4 = axes[1, 1]
for nombre, (lat, lon) in sitios.items():
    ax4.plot(lon, lat, 'ro', markersize=10)
    ax4.annotate(nombre, (lon, lat), xytext=(5, 5), textcoords='offset points')
ax4.set_xlabel('Longitud')
ax4.set_ylabel('Latitud')
ax4.set_title('Sitios de Análisis')
ax4.set_xlim(-106, -101)
ax4.set_ylim(17, 22)
ax4.grid(True, alpha=0.3)
ax4.set_aspect('equal')

plt.tight_layout()
plt.savefig('seismex_demo_resultados.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Resultados guardados en 'seismex_demo_resultados.png'")

---

## Resumen

Este notebook demostró las principales capacidades de SEISMEX:

1. **Análisis Gutenberg-Richter**: Estimación de parámetros de sismicidad
2. **Generación de Isosistas**: Mapas de intensidad con múltiples GMPEs/IPEs
3. **Modelos de Fuentes**: Definición de zonas sismogénicas
4. **Análisis PSHA**: Curvas de peligro y desagregación
5. **Optimización NSGA-II**: Ubicación óptima de infraestructura

Para más información, consulta la documentación en https://github.com/sebastiangz/seismex